In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

In [2]:
class GCNLinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCNLinkPredictor, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.fc = nn.Linear(out_channels * 2, 1)  # For concatenation

    def forward(self, x, edge_index, edge_pairs):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)

        # Extract edge embeddings
        edge_embeddings = torch.cat([x[edge_pairs[0]], x[edge_pairs[1]]], dim=1)
        scores = self.fc(edge_embeddings)
        return torch.sigmoid(scores)

In [3]:
# Example usage
num_nodes = 100
num_node_features = 16
hidden_channels = 32
out_channels = 16

In [10]:
# Randomly generated example data
node_features = torch.randn(num_nodes, num_node_features)
edges = torch.randint(0, num_nodes, (2, 200))
edge_pairs = torch.randint(0, num_nodes, (2, 50))

In [6]:
node_features.shape

torch.Size([100, 16])

In [9]:
edges.shape, edge_pairs.shape

(torch.Size([2, 200]), torch.Size([2, 50]))

In [12]:
data = Data(x=node_features, edge_index=edges)

In [13]:
model = GCNLinkPredictor(num_node_features, hidden_channels, out_channels)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

In [14]:
# Training loop
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    output = model(data.x, data.edge_index, edge_pairs)
    labels = torch.randint(0, 2, (50, 1)).float()  # Example labels
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 0.750506579875946
Epoch 2, Loss: 0.6855705976486206
Epoch 3, Loss: 0.7078419327735901
Epoch 4, Loss: 0.6960816979408264
Epoch 5, Loss: 0.6795642971992493
Epoch 6, Loss: 0.6894521117210388
Epoch 7, Loss: 0.6950029730796814
Epoch 8, Loss: 0.6779126524925232
Epoch 9, Loss: 0.7088271975517273
Epoch 10, Loss: 0.7344977855682373
